APP

In [5]:
from pyneuphonic import Neuphonic, TTSConfig, Agent
from pyneuphonic.player import AudioPlayer
import pyaudio
import asyncio

# PLACE YOUR API KEY HERE
api_key = "1701b983c6aa9def986084fad58939215af401e941c4fe219cf2e35780d805ed.c9441799-387a-4c62-979d-166ee460a801" # GET THIS FROM beta.neuphonic.com!!!!!!!!!

In [7]:
import asyncio
import aioconsole
import os

from pyneuphonic.client import Neuphonic
from pyneuphonic.models import APIResponse, AgentResponse, AgentConfig, WebsocketEvents
from pyneuphonic.player import AsyncAudioPlayer, AsyncAudioRecorder

def save_transcript(text, section):
    try:
        with open(f'transcripts/{section}.txt', 'a') as f:
            f.write(f'{text}\n')

    except FileNotFoundError:
        os.makedirs('transcripts', exist_ok=True)
        with open(f'transcripts/{section}.txt', 'a') as f:
            f.write(f'{text}\n')

def default_on_message(message: APIResponse[AgentResponse]):
    """
    Default callback function to handle messages from the server.

    Parameters
    ----------
    message : APIResponse[AgentResponse]
        The message received from the server, containing the type and text.
    """
    if message.data.type == 'user_transcript':
        print(f'User: {message.data.text}')
        if 'work items' in message.data.text.lower():
            save_transcript(message.data.text, 'work_items')
        elif 'cooking' in message.data.text.lower():
            save_transcript(message.data.text, 'cooking')
        else:
            save_transcript(message.data.text, 'quick_note')
    elif message.data.type == 'llm_response':
        print(f'Agent: {message.data.text}')


class Agent:
    def __init__(
        self, client: Neuphonic, mute=False, on_message=default_on_message, **kwargs
    ):
        """
        Initialize an Agent instance.

        Parameters
        ----------
        client : Neuphonic
            The Neuphonic client instance.
        mute : bool, optional
            If True, the agent will not play audio responses. Default is False.
        on_message : callable, optional
            A callback function to handle messages from the server. Default is default_on_message.
        **kwargs
            Additional keyword arguments to configure the agent. See the `AgentConfig` model for a
            full list of agent configuration parameters.
        """
        self.config = AgentConfig(**kwargs)
        self.mute = mute

        self.ws = client.agents.AsyncWebsocketClient()

        self.player = None
        if not self.mute:
            self.player = AsyncAudioPlayer()

        if 'asr' in self.config.mode:
            # passing in the websocket object will automatically forward audio to the server
            self.recorder = AsyncAudioRecorder(
                sampling_rate=self.config.incoming_sampling_rate,
                websocket=self.ws,
                player=self.player,
            )

        self.on_message_hook = on_message

    async def on_message(self, message: APIResponse[AgentResponse]):
        """
        Handle incoming messages from the server.

        Parameters
        ----------
        message : APIResponse[AgentResponse]
            The message received from the server, containing the type and content.
        """
        # server will return 3 types of messages: audio_response, user_transcript, llm_response
        if message.data.type == 'audio_response':
            if not self.mute:
                await self.player.play(message.data.audio)

        if message.data.type == 'llm_response':
            #print(1)
            if 'recorded' in message.data.text.lower():
                #print(2)
                await self.ws.close()
                await self.recorder.close()
                await self.player.close()

        if self.on_message_hook is not None and callable(self.on_message_hook):
            self.on_message_hook(message)

    async def start(self):
        """
        Start the agent, opening necessary connections and handling user input.
        """
        self.ws.on(WebsocketEvents.MESSAGE, self.on_message)
        self.ws.on(WebsocketEvents.CLOSE, self.on_close)

        if not self.mute:
            await self.player.open()
        await self.ws.open(self.config)

        if 'asr' in self.config.mode:
            await self.recorder.record()

            try:
                while True:
                    await asyncio.sleep(0.01)
            except KeyboardInterrupt:
                await self.ws.close()

        else:
            while True:
                user_text = await aioconsole.ainput(
                    "\nEnter text to speak (or 'quit' to exit): "
                )

                if user_text.lower() == 'quit':
                    break

                await self.ws.send({'text': user_text})
                await asyncio.sleep(1)  # simply for formatting

    async def on_close(self):
        """
        Handle the closing of connections and cleanup resources.
        """
        if not self.mute:
            await self.player.close()
        if 'asr' in self.config.mode:
            await self.recorder.close()

In [3]:
client = Neuphonic(api_key=api_key)

agent_id = client.agents.create(
    name='Agent',
    prompt='You are a helpful agent. I will give you an idea to record. You have to listen to the full idea that I am telling you. Then I will stop the idea by saying that is all. After I tell you the idea. Say thank you, recorded.',
    greeting='I am ready to record your ideas.'
).data['id']

agent = Agent(client, agent_id=agent_id, tts_model='neu_hq')

await agent.start()

User: Today remind me that I've got to go to the Cambridge hackathon.


ERROR:root:Error in _send: sent 1000 (OK); then received 1000 (OK)
ERROR:root:Error in _send: sent 1000 (OK); then received 1000 (OK)
ERROR:root:Error in _send: sent 1000 (OK); then received 1000 (OK)
ERROR:root:Error in _send: sent 1000 (OK); then received 1000 (OK)
ERROR:root:Error in _send: sent 1000 (OK); then received 1000 (OK)
ERROR:root:Error in _send: sent 1000 (OK); then received 1000 (OK)


Agent: Thank you for letting me know. I have recorded that you need to go to the Cambridge hackathon. Is there anything else I can assist you with?


CancelledError: 